# Caracal RL cyber (Kaggle T4) — GRPO direto + RSI, um notebook

**Settings**: Accelerator -> **GPU T4 x2** · Internet **ON** · Save Version -> Run All (batch).

Roda os dois modos, escolhido por `MODE` na celula 1:
- `MODE='grpo'` — RL direto (`B_grpo_straight`): todo o compute em treino. RODAR ESTE PRIMEIRO.
- `MODE='rsi'`  — outer loop same-model (`C_rsi_outer`): propoe mutacao, treina, retem se melhora.

**Regua**: base = step 0 do run (~44%). Piso de colapso (sempre CWE-79) = 0.273.
Alvo = Foundation-Sec-8B RCM 72-75. `unparsed` alto = truncagem apertou, nao o modelo piorando.

**Quota**: cada conta Kaggle tem 30h/semana de GPU. Se a sua esgotou, rode em
outra conta de founder — o notebook clona tudo do GitHub, nao depende do disco local.

In [ ]:
MODE = 'grpo'          # 'grpo' (rodar primeiro) ou 'rsi'
TOTAL_STEPS = 120      # grpo: total de steps de RL
GENS, CANDS, STEPS = 2, 2, 10   # rsi: geracoes x candidatos x 10 steps/candidato
print('MODE =', MODE)

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), 'Sem GPU: Settings -> Accelerator -> GPU T4 x2'
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv']).decode())

In [ ]:
# Stack pinado (torch 2.6 cu124, sem Unsloth — T4 e SM 7.5). torchao removido.
!pip -q install 'torch==2.6.0' 'torchvision==0.21.0' --index-url https://download.pytorch.org/whl/cu124
!pip -q install 'transformers==4.49.0' 'peft==0.14.0' 'trl==0.15.2' 'accelerate>=1.0.0' 'datasets>=3.0.0' 'sentence-transformers' 'scipy' 'statsmodels' 'antlr4-python3-runtime==4.11'
!pip -q uninstall -y torchao 2>/dev/null
import transformers, torch
print('transformers', transformers.__version__, '| torch', torch.__version__)

In [ ]:
# Codigo com todos os fixes + dataset CVE->CWE
import os, subprocess
if not os.path.exists('/kaggle/working/caracal-1'):
    subprocess.run(['git','clone','--depth','1','-b','s07-hybrid-agentic',
                    'https://github.com/iterate-labs-ai/caracal-1.git',
                    '/kaggle/working/caracal-1'], check=True)
os.chdir('/kaggle/working/caracal-1')
print(subprocess.check_output(['git','rev-parse','--short','HEAD']).decode().strip())
subprocess.run(['python','-m','data.ignite.build_cyber_rcm'], check=True)

In [ ]:
import sys, os
BASE = 'Qwen/Qwen2.5-Coder-3B-Instruct'
OUT = f'/kaggle/working/rl_cyber_{MODE}'
env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')

if MODE == 'grpo':
    cmd = [sys.executable,'-m','train.ignite.B_grpo_straight',
           '--base',BASE,'--out',OUT,'--total-steps',str(TOTAL_STEPS),
           '--block','20','--lr','1e-6','--rank','32','--eval-n','150']
    if os.path.exists(f'{OUT}/curve.json'): cmd.append('--resume')
elif MODE == 'rsi':
    cmd = [sys.executable,'-m','train.ignite.C_rsi_outer',
           '--base',BASE,'--bench','cyber_rcm','--bench-name','cyber_rcm',
           '--dataset-train','data/ignite/cyber_rcm_train.jsonl',
           '--dataset-dev','data/ignite/cyber_rcm_dev.jsonl',
           '--dataset-val','data/ignite/cyber_rcm_val.jsonl',
           '--gens',str(GENS),'--cands',str(CANDS),'--steps',str(STEPS),'--out',OUT]
else:
    raise ValueError(MODE)
print(' '.join(cmd), flush=True)
subprocess.run(cmd, check=True, env=env)

In [ ]:
# BENCHMARK do resultado no dev (base vs cada adapter produzido)
import json, glob, gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from eval.ignite.benches import BENCH_REGISTRY

DEV = 'data/ignite/cyber_rcm_dev.jsonl'
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = 'left'

def bench(adapter):
    m = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map='cuda')
    if adapter: m = PeftModel.from_pretrained(m, adapter)
    m.eval()
    r = BENCH_REGISTRY['cyber_rcm'](m, tok, n=150, dataset_path=DEV)
    m = None; gc.collect(); torch.cuda.empty_cache()
    return r['accuracy'], r['hier_score'], r['unparsed_frac']

if MODE == 'grpo' and os.path.exists(f'{OUT}/curve.json'):
    curve = json.load(open(f'{OUT}/curve.json'))
    print('=== CURVA GRPO (base=step0 | piso de colapso CWE-79=0.273) ===')
    for s,v in sorted(curve.items(), key=lambda x:int(x[0])):
        print(f"  step {int(s):3d}: acc={v['accuracy']:.3f} hier={v['hier']:.3f} unparsed={v['unparsed']:.2f}")
    base=curve['0']['accuracy']; best=max(v['accuracy'] for v in curve.values())
    print(f"\nbase {base:.3f} -> melhor {best:.3f}  (delta {best-base:+.3f})")
else:
    # RSI: benchmarka base + todo adapter salvo (gen*/cand*), ordenado por acc
    print('=== BENCHMARK RSI no dev (n=150 | piso de colapso=0.273) ===', flush=True)
    b_acc, b_hier, b_un = bench(None)
    print(f"  BASE          acc={b_acc:.3f} hier={b_hier:.3f} unparsed={b_un:.2f}", flush=True)
    rows=[]
    for ad in sorted(glob.glob(f'{OUT}/gen*/cand*')):
        if not glob.glob(ad+'/adapter_*.safetensors') and not glob.glob(ad+'/adapter_*.bin'): continue
        a,h,u = bench(ad)
        tag = ad.split('/',)[-2]+'/'+ad.split('/')[-1]
        rows.append((a, tag, h, u))
        print(f"  {tag:14s} acc={a:.3f} hier={h:.3f} unparsed={u:.2f}  (vs base {a-b_acc:+.3f})", flush=True)
    if rows:
        best=max(rows)
        print(f"\nmelhor: {best[1]} acc={best[0]:.3f} (delta vs base {best[0]-b_acc:+.3f})")